## PACOTES 

In [1]:
import os
import time

import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy.stats import ks_2samp

from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_recall_curve,
    auc,
    f1_score,
    matthews_corrcoef,
    log_loss,
    confusion_matrix
)

from openTSNE import TSNE

## CONF GERAIS

In [2]:
# BASE
RANK = 1
TARGET_COL_D = "status_fraude"
THRESHOLD_D = 0.50
estado_randomico_D = 42
NOME_HTML_D = f"3d_rank_{RANK}_tsne.html"

# HIPERPARÂMETROS GMM
numero_de_componentes_D = 2
inicializacoes_gausianas_D = 3
tipo_matriz_covariancia_D = "full"
erro_numerico_D = 1e-6

# HIPERPARÂMETROS t-SNE
TSNE_PERPLEXITY_D = 30
TSNE_N_ITER_D = 1500
TSNE_EARLY_EXAGGERATION_ITER_D = 100
TSNE_EARLY_EXAGGERATION_D = 12
TSNE_EXAGGERATION_D = 1
TSNE_LEARNING_RATE_D = "auto"
TSNE_METRIC_D = "euclidean"
TSNE_INITIALIZATION_D = "pca"
TSNE_NEGATIVE_GRADIENT_METHOD_D = "bh"
TSNE_N_JOBS_D = 7
TSNE_RANDOM_STATE_D = estado_randomico_D
TSNE_VERBOSE_D = True


## FUNCAO SCORE 

In [3]:
def calcular_score_final():
    auc_pr_norm = np.clip(auc_pr, 0, 1)
    mcc_norm = (mcc + 1) / 2
    mcc_norm = np.clip(mcc_norm, 0, 1)
    ks_norm = np.clip(ks, 0, 1)
    log_loss_norm = 1 / (1 + ll)
    score = (
        mcc_norm +
        ks_norm +
        log_loss_norm +
        auc_pr_norm
    ) / 4

    return round(float(score), 6)


## FUNCAO T-SNE/ORIGINAL 

In [4]:
def gerar_relatorio_3d(
    RANK=1,
    usar_tsne=True,
    TARGET_COL="status_fraude",
    THRESHOLD=0.50,
    estado_randomico=42,

    numero_de_componentes=2,
    inicializacoes_gausianas=3,
    tipo_matriz_covariancia="full",
    erro_numerico=1e-6,

    TSNE_PERPLEXITY=30,
    TSNE_N_ITER=1000,
    TSNE_EARLY_EXAGGERATION_ITER=100,
    TSNE_EARLY_EXAGGERATION=12,
    TSNE_EXAGGERATION=1,
    TSNE_LEARNING_RATE="auto",
    TSNE_METRIC="euclidean",
    TSNE_INITIALIZATION="pca",
    TSNE_NEGATIVE_GRADIENT_METHOD="bh",
    TSNE_N_JOBS=7,
    TSNE_VERBOSE=True
):

    def calcular_score_final():
        auc_pr_norm = np.clip(auc_pr, 0, 1)

        mcc_norm = (mcc + 1) / 2
        mcc_norm = np.clip(mcc_norm, 0, 1)

        ks_norm = np.clip(ks, 0, 1)

        log_loss_norm = 1 / (1 + ll)

        score_final_calc = (
            auc_pr_norm +
            mcc_norm +
            ks_norm +
            log_loss_norm
        ) / 4

        return round(float(score_final_calc), 6)

    # NOME HTML
    tipo_relatorio = "tsne" if usar_tsne else "orig"
    NOME_HTML = f"3d_rank_{RANK}_{tipo_relatorio}.html"

    # DIRETÓRIO
    try:
        BASE_DIR = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        BASE_DIR = os.getcwd()

    HTML_PATH = os.path.join(BASE_DIR, NOME_HTML)

    print("Diretório:", BASE_DIR)

    # LOAD RANKING
    df_scores = pd.read_csv("3x3_visu_scores.csv")

    row = df_scores[
        df_scores["Posicao_Rank"] == RANK
    ].iloc[0]

    feature_1 = row["Feature_1"]
    feature_2 = row["Feature_2"]
    feature_3 = row["Feature_3"]

    print(f"\nRank Selecionado: {RANK}")
    print(f"Features: {feature_1} vs {feature_2} vs {feature_3}")

    # LOAD DATASET
    df = pd.read_csv("creditcard.csv")

    # BASE COMPLETA
    df_model = df[
        [feature_1, feature_2, feature_3, TARGET_COL]
    ].dropna().reset_index(drop=True)

    print("\nQuantidade usada:")
    print(df_model[TARGET_COL].value_counts())

    # FEATURES ORIGINAIS
    X_original = df_model[
        [feature_1, feature_2, feature_3]
    ]

    y = df_model[TARGET_COL]

    # SE USAR t-SNE
    if usar_tsne:

        print("\nRodando t-SNE 3D...\n")

        scaler_original = StandardScaler()

        X_scaled_original = scaler_original.fit_transform(
            X_original
        )

        inicio_tsne = time.perf_counter()

        tsne = TSNE(
            n_components=3,
            perplexity=TSNE_PERPLEXITY,
            learning_rate=TSNE_LEARNING_RATE,
            early_exaggeration_iter=TSNE_EARLY_EXAGGERATION_ITER,
            early_exaggeration=TSNE_EARLY_EXAGGERATION,
            n_iter=TSNE_N_ITER,
            exaggeration=TSNE_EXAGGERATION,
            metric=TSNE_METRIC,
            initialization=TSNE_INITIALIZATION,
            negative_gradient_method=TSNE_NEGATIVE_GRADIENT_METHOD,
            n_jobs=TSNE_N_JOBS,
            random_state=estado_randomico,
            verbose=TSNE_VERBOSE
        )

        X_tsne = tsne.fit(X_scaled_original)
        X_tsne = np.asarray(X_tsne)

        fim_tsne = time.perf_counter()

        print(
            f"\nt-SNE 3D finalizado em "
            f"{(fim_tsne - inicio_tsne):.2f} segundos."
        )

        df_model["TSNE_1"] = X_tsne[:, 0]
        df_model["TSNE_2"] = X_tsne[:, 1]
        df_model["TSNE_3"] = X_tsne[:, 2]

        cols_modelo = ["TSNE_1", "TSNE_2", "TSNE_3"]

        titulo_corr = "Correlação Spearman - t-SNE 3D"
        titulo_scatter = "Distribuição 3D após t-SNE"
        titulo_prob = "Scatter 3D - Probabilidade GMM após t-SNE"
        titulo_relatorio = "Relatório GMM após t-SNE 3D (Spearman)"

    else:

        cols_modelo = [feature_1, feature_2, feature_3]

        titulo_corr = "Correlação Spearman - Features Originais"
        titulo_scatter = "Distribuição 3D das Features Originais"
        titulo_prob = "Scatter 3D - Probabilidade GMM nas Features Originais"
        titulo_relatorio = "Relatório GMM 3D - Features Originais"

    # FEATURES PARA GMM
    X = df_model[cols_modelo]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # GMM
    gmm = GaussianMixture(
        n_components=numero_de_componentes,
        covariance_type=tipo_matriz_covariancia,
        random_state=estado_randomico,
        reg_covar=erro_numerico,
        n_init=inicializacoes_gausianas
    )

    gmm.fit(X_scaled)

    # CLUSTERS
    clusters = gmm.predict(X_scaled)

    ct = pd.crosstab(clusters, y)

    print("\nTabela Cluster x Classe Real:")
    print(ct)

    if 1 not in ct.columns:
        raise ValueError("Nenhuma fraude encontrada nos clusters.")

    cluster_fraude = ct[1].idxmax()

    print(f"\nCluster identificado como fraude: {cluster_fraude}")

    # PROBABILIDADES
    score = gmm.predict_proba(X_scaled)[:, cluster_fraude]

    score = np.clip(
        score,
        1e-15,
        1 - 1e-15
    )

    df_model["Probabilidade_GMM"] = score

    # PREDIÇÃO
    y_pred = (score >= THRESHOLD).astype(int)

    # MÉTRICAS — MESMAS DO 2D
    prec, rec, _ = precision_recall_curve(y, score)

    auc_pr = auc(rec, prec)

    mcc = matthews_corrcoef(y, y_pred)

    ks = ks_2samp(
        score[y == 0],
        score[y == 1]
    ).statistic

    ll = log_loss(y, score)

    score_final = calcular_score_final()

    # MATRIZ CONFUSÃO
    cm = confusion_matrix(
        y,
        y_pred,
        labels=[0, 1]
    )

    cm_percent = (
        cm.astype(float)
        / cm.sum(axis=1)[:, np.newaxis]
    ) * 100

    texto_cm = []

    for i in range(2):

        linha = []

        for j in range(2):

            linha.append(
                f"{cm_percent[i, j]:.2f}%"
                f"<br>({cm[i, j]})"
            )

        texto_cm.append(linha)

    # CORRELAÇÃO SPEARMAN
    corr = df_model[
        cols_modelo
    ].corr(method="spearman")

    # DATASETS SCATTER
    df_fraude = df_model[
        df_model[TARGET_COL] == 1
    ]

    df_nao_fraude = df_model[
        df_model[TARGET_COL] == 0
    ]

    # FIGURA
    fig = make_subplots(
        rows=4,
        cols=2,

        specs=[
            [
                {"type": "heatmap"},
                {"type": "heatmap"}
            ],
            [
                {"colspan": 2},
                None
            ],
            [
                {"type": "scatter3d", "colspan": 2},
                None
            ],
            [
                {"type": "scatter3d", "colspan": 2},
                None
            ]
        ],

        row_heights=[
            0.20,
            0.12,
            0.34,
            0.34
        ],

        horizontal_spacing=0.10,
        vertical_spacing=0.075,

        subplot_titles=(
            titulo_corr,
            "Matriz de Confusão (%)",
            "",
            titulo_scatter,
            titulo_prob
        )
    )

    # HEATMAP CORRELAÇÃO
    fig.add_trace(
        go.Heatmap(
            z=corr.values,
            x=cols_modelo,
            y=cols_modelo,
            text=np.round(corr.values, 3),
            texttemplate="%{text}",
            textfont=dict(size=18),
            colorscale="RdBu",
            zmin=-1,
            zmax=1,
            showscale=False
        ),

        row=1,
        col=1
    )

    # HEATMAP CONFUSÃO
    fig.add_trace(
        go.Heatmap(
            z=cm_percent,
            x=[
                "Pred Não Fraude",
                "Pred Fraude"
            ],
            y=[
                "Real Não Fraude",
                "Real Fraude"
            ],
            text=texto_cm,
            texttemplate="%{text}",
            textfont=dict(size=18),
            colorscale="Blues",
            zmin=0,
            zmax=100,
            showscale=False
        ),

        row=1,
        col=2
    )

    # MÉTRICAS — MESMO BLOCO DO 2D
    metricas = f"""
<b>MÉTRICAS</b><br><br>
AUC-PR: {auc_pr:.4f}<br>
MCC: {mcc:.4f}<br>
KS: {ks:.4f}<br>
Log Loss: {ll:.4f}<br>
Score Final: {score_final:.4f}
"""

    fig.add_trace(
        go.Scatter(
            x=[0.5],
            y=[0.5],
            mode="text",
            text=[metricas],
            textfont=dict(size=20),
            showlegend=False
        ),

        row=2,
        col=1
    )

    fig.update_xaxes(
        visible=False,
        range=[0, 1],
        row=2,
        col=1
    )

    fig.update_yaxes(
        visible=False,
        range=[0, 1],
        row=2,
        col=1
    )

    # SCATTER 3D NÃO FRAUDE - CLASSES REAIS
    fig.add_trace(
        go.Scatter3d(
            x=df_nao_fraude[cols_modelo[0]],
            y=df_nao_fraude[cols_modelo[1]],
            z=df_nao_fraude[cols_modelo[2]],

            mode="markers",

            name="Não Fraude",

            marker=dict(
                color="rgba(0,0,255,0.18)",
                size=2
            )
        ),

        row=3,
        col=1
    )

    # SCATTER 3D FRAUDE - CLASSES REAIS
    fig.add_trace(
        go.Scatter3d(
            x=df_fraude[cols_modelo[0]],
            y=df_fraude[cols_modelo[1]],
            z=df_fraude[cols_modelo[2]],

            mode="markers",

            name="Fraude",

            marker=dict(
                color="rgba(255,0,0,0.95)",
                size=4
            )
        ),

        row=3,
        col=1
    )

    # SCATTER 3D PROBABILIDADE GMM
    fig.add_trace(
        go.Scatter3d(
            x=df_model[cols_modelo[0]],
            y=df_model[cols_modelo[1]],
            z=df_model[cols_modelo[2]],

            mode="markers",

            name="Probabilidade GMM",

            marker=dict(
                color=df_model["Probabilidade_GMM"],
                colorscale="Turbo",
                cmin=0,
                cmax=1,
                size=2,
                opacity=0.65,

                colorbar=dict(
                    title=dict(
                        text="Probabilidade de Fraude",
                        side="bottom",
                        font=dict(size=16)
                    ),
                    orientation="h",
                    x=0.5,
                    xanchor="center",
                    y=-0.10,
                    yanchor="top",
                    len=0.70,
                    thickness=22,
                    tickfont=dict(size=14)
                )
            ),

            showlegend=False
        ),

        row=4,
        col=1
    )

    # DESTACAR FRAUDES NO MAPA DE PROBABILIDADE
    fig.add_trace(
        go.Scatter3d(
            x=df_fraude[cols_modelo[0]],
            y=df_fraude[cols_modelo[1]],
            z=df_fraude[cols_modelo[2]],

            mode="markers",

            name="Fraudes sobre mapa GMM",

            marker=dict(
                color="rgba(255,0,0,0.95)",
                size=4,
                line=dict(
                    color="white",
                    width=1
                )
            ),

            showlegend=False
        ),

        row=4,
        col=1
    )

    # SCENE 3D - CLASSES REAIS
    fig.update_scenes(
        xaxis_title=cols_modelo[0],
        yaxis_title=cols_modelo[1],
        zaxis_title=cols_modelo[2],

        xaxis=dict(
            backgroundcolor="white",
            gridcolor="lightgray",
            zerolinecolor="lightgray"
        ),

        yaxis=dict(
            backgroundcolor="white",
            gridcolor="lightgray",
            zerolinecolor="lightgray"
        ),

        zaxis=dict(
            backgroundcolor="white",
            gridcolor="lightgray",
            zerolinecolor="lightgray"
        ),

        camera=dict(
            eye=dict(
                x=1.7,
                y=1.7,
                z=1.2
            )
        ),

        row=3,
        col=1
    )

    # SCENE 3D - PROBABILIDADE GMM
    fig.update_scenes(
        xaxis_title=cols_modelo[0],
        yaxis_title=cols_modelo[1],
        zaxis_title=cols_modelo[2],

        xaxis=dict(
            backgroundcolor="white",
            gridcolor="lightgray",
            zerolinecolor="lightgray"
        ),

        yaxis=dict(
            backgroundcolor="white",
            gridcolor="lightgray",
            zerolinecolor="lightgray"
        ),

        zaxis=dict(
            backgroundcolor="white",
            gridcolor="lightgray",
            zerolinecolor="lightgray"
        ),

        camera=dict(
            eye=dict(
                x=1.7,
                y=1.7,
                z=1.2
            )
        ),

        row=4,
        col=1
    )

    # LAYOUT
    if usar_tsne:
        descricao_features = f"""
        Features originais:
        {feature_1} vs {feature_2} vs {feature_3}
        <br>
        Novas features:
        TSNE_1 vs TSNE_2 vs TSNE_3
        """
    else:
        descricao_features = f"""
        Features originais:
        {feature_1} vs {feature_2} vs {feature_3}
        """

    fig.update_layout(
        title=dict(
            text=f"""
            {titulo_relatorio}
            <br>
            Rank {RANK}
            <br>
            {descricao_features}
            <br>
            Base completa: 100% fraudes + 100% não fraudes
            <br>
            Corte: Probabilidade Cluster Fraude ≥ {THRESHOLD:.2f}
            """,

            x=0.5,
            y=0.985,

            xanchor="center",
            yanchor="top",

            font=dict(size=26)
        ),

        width=1900,
        height=3000,

        template="plotly_white",

        font=dict(size=18),

        margin=dict(
            t=380,
            b=280,
            l=120,
            r=120
        ),

        legend=dict(
            orientation="h",
            font=dict(size=18),
            yanchor="bottom",
            y=0.01,
            xanchor="center",
            x=0.5
        )
    )

    # SAVE HTML
    fig.write_html(
        HTML_PATH,
        include_plotlyjs="cdn"
    )

    print("\nHTML GERADO COM SUCESSO:")
    print(HTML_PATH)

    return {
        "HTML_PATH": HTML_PATH,
        "Rank": RANK,
        "Tipo": tipo_relatorio,
        "Features": [feature_1, feature_2, feature_3],
        "AUC_PR": auc_pr,
        "MCC": mcc,
        "KS": ks,
        "Log_Loss": ll,
        "Score_Final": score_final
    }

## 1ST COLOCADO

### T-SNE

In [5]:
resultado = gerar_relatorio_3d(
    RANK=1,
    usar_tsne=True,
    TARGET_COL= TARGET_COL_D,
    THRESHOLD= THRESHOLD_D,
    estado_randomico= estado_randomico_D,

    numero_de_componentes= numero_de_componentes_D,
    inicializacoes_gausianas= inicializacoes_gausianas_D,
    tipo_matriz_covariancia= tipo_matriz_covariancia_D,
    erro_numerico= erro_numerico_D,

    TSNE_PERPLEXITY= TSNE_PERPLEXITY_D,
    TSNE_N_ITER= TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER= TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION= TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION= TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC= TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 1
Features: V11 vs V17 vs V22

Quantidade usada:
status_fraude
0    284315
1       492
Name: count, dtype: int64

Rodando t-SNE 3D...

--------------------------------------------------------------------------------
TSNE(early_exaggeration=12, early_exaggeration_iter=100, exaggeration=1,
     n_components=3, n_iter=1500, n_jobs=7, negative_gradient_method='bh',
     random_state=42, verbose=True)
--------------------------------------------------------------------------------
===> Finding 90 nearest neighbors using Annoy approximate search using euclidean distance...
   --> Time elapsed: 108.54 seconds
===> Calculating affinity matrix...
   --> Time elapsed: 91.72 seconds
===> Calculating PCA-based initialization...
   --> Time elapsed: 0.57 seconds
===> Running optimization with exaggeration=12.00, lr=23733.92 for 100 iterations...
Iteration   

### ORIGINAL

In [6]:
resultado = gerar_relatorio_3d(
    RANK=1,
    usar_tsne=False,
    TARGET_COL= TARGET_COL_D,
    THRESHOLD= THRESHOLD_D,
    estado_randomico= estado_randomico_D,

    numero_de_componentes= numero_de_componentes_D,
    inicializacoes_gausianas= inicializacoes_gausianas_D,
    tipo_matriz_covariancia= tipo_matriz_covariancia_D,
    erro_numerico= erro_numerico_D,

    TSNE_PERPLEXITY= TSNE_PERPLEXITY_D,
    TSNE_N_ITER= TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER= TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION= TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION= TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC= TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 1
Features: V11 vs V17 vs V22

Quantidade usada:
status_fraude
0    284315
1       492
Name: count, dtype: int64

Tabela Cluster x Classe Real:
status_fraude       0    1
row_0                     
0                4620  418
1              279695   74

Cluster identificado como fraude: 0

HTML GERADO COM SUCESSO:
c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\3d_rank_1_orig.html
{'HTML_PATH': 'c:\\Users\\Vitor Craveiro\\Desktop\\UNESP\\MATÉRIAS FACULDADES\\MATÉRIAS 12 SEMESTRES\\TRABALHO DE CONCLUSAO DE CURSO 2\\3d_rank_1_orig.html', 'Rank': 1, 'Tipo': 'orig', 'Features': ['V11', 'V17', 'V22'], 'AUC_PR': 0.5608017146116053, 'MCC': 0.26252932920781247, 'KS': np.float64(0.8509775957017788), 'Log_Loss': 0.11182654581025873, 'Score_Final': 0.735616}


## 2ND COLOCADO

### T-SNE

In [7]:
resultado = gerar_relatorio_3d(
    RANK=2,
    usar_tsne=True,
    TARGET_COL= TARGET_COL_D,
    THRESHOLD= THRESHOLD_D,
    estado_randomico= estado_randomico_D,

    numero_de_componentes= numero_de_componentes_D,
    inicializacoes_gausianas= inicializacoes_gausianas_D,
    tipo_matriz_covariancia= tipo_matriz_covariancia_D,
    erro_numerico= erro_numerico_D,

    TSNE_PERPLEXITY= TSNE_PERPLEXITY_D,
    TSNE_N_ITER= TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER= TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION= TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION= TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC= TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 2
Features: V11 vs V15 vs V17

Quantidade usada:
status_fraude
0    284315
1       492
Name: count, dtype: int64

Rodando t-SNE 3D...

--------------------------------------------------------------------------------
TSNE(early_exaggeration=12, early_exaggeration_iter=100, exaggeration=1,
     n_components=3, n_iter=1500, n_jobs=7, negative_gradient_method='bh',
     random_state=42, verbose=True)
--------------------------------------------------------------------------------
===> Finding 90 nearest neighbors using Annoy approximate search using euclidean distance...
   --> Time elapsed: 170.91 seconds
===> Calculating affinity matrix...
   --> Time elapsed: 48.71 seconds
===> Calculating PCA-based initialization...
   --> Time elapsed: 0.13 seconds
===> Running optimization with exaggeration=12.00, lr=23733.92 for 100 iterations...
Iteration   

### ORIGINAL 

In [8]:
resultado = gerar_relatorio_3d(
    RANK=2,
    usar_tsne=False,
    TARGET_COL= TARGET_COL_D,
    THRESHOLD= THRESHOLD_D,
    estado_randomico= estado_randomico_D,

    numero_de_componentes= numero_de_componentes_D,
    inicializacoes_gausianas= inicializacoes_gausianas_D,
    tipo_matriz_covariancia= tipo_matriz_covariancia_D,
    erro_numerico= erro_numerico_D,

    TSNE_PERPLEXITY= TSNE_PERPLEXITY_D,
    TSNE_N_ITER= TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER= TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION= TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION= TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC= TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 2
Features: V11 vs V15 vs V17

Quantidade usada:
status_fraude
0    284315
1       492
Name: count, dtype: int64

Tabela Cluster x Classe Real:
status_fraude       0    1
row_0                     
0                5854  419
1              278461   73

Cluster identificado como fraude: 0

HTML GERADO COM SUCESSO:
c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\3d_rank_2_orig.html
{'HTML_PATH': 'c:\\Users\\Vitor Craveiro\\Desktop\\UNESP\\MATÉRIAS FACULDADES\\MATÉRIAS 12 SEMESTRES\\TRABALHO DE CONCLUSAO DE CURSO 2\\3d_rank_2_orig.html', 'Rank': 2, 'Tipo': 'orig', 'Features': ['V11', 'V15', 'V17'], 'AUC_PR': 0.5712291239203129, 'MCC': 0.23513951409771774, 'KS': np.float64(0.8506091591700434), 'Log_Loss': 0.13231336866275786, 'Score_Final': 0.730639}


## 3RD COLOCADO

### T-SNE

In [9]:
resultado = gerar_relatorio_3d(
    RANK=3,
    usar_tsne=True,
    TARGET_COL= TARGET_COL_D,
    THRESHOLD= THRESHOLD_D,
    estado_randomico= estado_randomico_D,

    numero_de_componentes= numero_de_componentes_D,
    inicializacoes_gausianas= inicializacoes_gausianas_D,
    tipo_matriz_covariancia= tipo_matriz_covariancia_D,
    erro_numerico= erro_numerico_D,

    TSNE_PERPLEXITY= TSNE_PERPLEXITY_D,
    TSNE_N_ITER= TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER= TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION= TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION= TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC= TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 3
Features: V11 vs V17 vs V26

Quantidade usada:
status_fraude
0    284315
1       492
Name: count, dtype: int64

Rodando t-SNE 3D...

--------------------------------------------------------------------------------
TSNE(early_exaggeration=12, early_exaggeration_iter=100, exaggeration=1,
     n_components=3, n_iter=1500, n_jobs=7, negative_gradient_method='bh',
     random_state=42, verbose=True)
--------------------------------------------------------------------------------
===> Finding 90 nearest neighbors using Annoy approximate search using euclidean distance...
   --> Time elapsed: 135.23 seconds
===> Calculating affinity matrix...
   --> Time elapsed: 36.42 seconds
===> Calculating PCA-based initialization...
   --> Time elapsed: 0.09 seconds
===> Running optimization with exaggeration=12.00, lr=23733.92 for 100 iterations...
Iteration   

### ORIGINAL 

In [10]:
resultado = gerar_relatorio_3d(
    RANK=3,
    usar_tsne=False,
    TARGET_COL= TARGET_COL_D,
    THRESHOLD= THRESHOLD_D,
    estado_randomico= estado_randomico_D,

    numero_de_componentes= numero_de_componentes_D,
    inicializacoes_gausianas= inicializacoes_gausianas_D,
    tipo_matriz_covariancia= tipo_matriz_covariancia_D,
    erro_numerico= erro_numerico_D,

    TSNE_PERPLEXITY= TSNE_PERPLEXITY_D,
    TSNE_N_ITER= TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER= TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION= TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION= TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC= TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 3
Features: V11 vs V17 vs V26

Quantidade usada:
status_fraude
0    284315
1       492
Name: count, dtype: int64

Tabela Cluster x Classe Real:
status_fraude       0    1
row_0                     
0              277913   71
1                6402  421

Cluster identificado como fraude: 1

HTML GERADO COM SUCESSO:
c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\3d_rank_3_orig.html
{'HTML_PATH': 'c:\\Users\\Vitor Craveiro\\Desktop\\UNESP\\MATÉRIAS FACULDADES\\MATÉRIAS 12 SEMESTRES\\TRABALHO DE CONCLUSAO DE CURSO 2\\3d_rank_3_orig.html', 'Rank': 3, 'Tipo': 'orig', 'Features': ['V11', 'V17', 'V26'], 'AUC_PR': 0.5756820505261391, 'MCC': 0.22626659729968787, 'KS': np.float64(0.8446602224230568), 'Log_Loss': 0.13456884414487966, 'Score_Final': 0.728717}


## 4TH COLOCADO

### T-SNE

In [11]:
resultado = gerar_relatorio_3d(
    RANK=4,
    usar_tsne=True,
    TARGET_COL= TARGET_COL_D,
    THRESHOLD= THRESHOLD_D,
    estado_randomico= estado_randomico_D,

    numero_de_componentes= numero_de_componentes_D,
    inicializacoes_gausianas= inicializacoes_gausianas_D,
    tipo_matriz_covariancia= tipo_matriz_covariancia_D,
    erro_numerico= erro_numerico_D,

    TSNE_PERPLEXITY= TSNE_PERPLEXITY_D,
    TSNE_N_ITER= TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER= TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION= TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION= TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC= TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 4
Features: V11 vs V13 vs V17

Quantidade usada:
status_fraude
0    284315
1       492
Name: count, dtype: int64

Rodando t-SNE 3D...

--------------------------------------------------------------------------------
TSNE(early_exaggeration=12, early_exaggeration_iter=100, exaggeration=1,
     n_components=3, n_iter=1500, n_jobs=7, negative_gradient_method='bh',
     random_state=42, verbose=True)
--------------------------------------------------------------------------------
===> Finding 90 nearest neighbors using Annoy approximate search using euclidean distance...
   --> Time elapsed: 82.48 seconds
===> Calculating affinity matrix...
   --> Time elapsed: 21.40 seconds
===> Calculating PCA-based initialization...
   --> Time elapsed: 0.03 seconds
===> Running optimization with exaggeration=12.00, lr=23733.92 for 100 iterations...
Iteration   5

### ORIGINAL 

In [12]:
resultado = gerar_relatorio_3d(
    RANK=4,
    usar_tsne=False,
    TARGET_COL= TARGET_COL_D,
    THRESHOLD= THRESHOLD_D,
    estado_randomico= estado_randomico_D,

    numero_de_componentes= numero_de_componentes_D,
    inicializacoes_gausianas= inicializacoes_gausianas_D,
    tipo_matriz_covariancia= tipo_matriz_covariancia_D,
    erro_numerico= erro_numerico_D,

    TSNE_PERPLEXITY= TSNE_PERPLEXITY_D,
    TSNE_N_ITER= TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER= TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION= TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION= TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC= TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 4
Features: V11 vs V13 vs V17

Quantidade usada:
status_fraude
0    284315
1       492
Name: count, dtype: int64

Tabela Cluster x Classe Real:
status_fraude       0    1
row_0                     
0                7092  423
1              277223   69

Cluster identificado como fraude: 0

HTML GERADO COM SUCESSO:
c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\3d_rank_4_orig.html
{'HTML_PATH': 'c:\\Users\\Vitor Craveiro\\Desktop\\UNESP\\MATÉRIAS FACULDADES\\MATÉRIAS 12 SEMESTRES\\TRABALHO DE CONCLUSAO DE CURSO 2\\3d_rank_4_orig.html', 'Rank': 4, 'Tipo': 'orig', 'Features': ['V11', 'V13', 'V17'], 'AUC_PR': 0.570066438794715, 'MCC': 0.21629073872556825, 'KS': np.float64(0.8460574331487648), 'Log_Loss': 0.14611492190486994, 'Score_Final': 0.724196}


## 5TH COLOCADO

### T-SNE

In [13]:
resultado = gerar_relatorio_3d(
    RANK=5,
    usar_tsne=True,
    TARGET_COL= TARGET_COL_D,
    THRESHOLD= THRESHOLD_D,
    estado_randomico= estado_randomico_D,

    numero_de_componentes= numero_de_componentes_D,
    inicializacoes_gausianas= inicializacoes_gausianas_D,
    tipo_matriz_covariancia= tipo_matriz_covariancia_D,
    erro_numerico= erro_numerico_D,

    TSNE_PERPLEXITY= TSNE_PERPLEXITY_D,
    TSNE_N_ITER= TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER= TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION= TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION= TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC= TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 5
Features: V14 vs V17 vs V26

Quantidade usada:
status_fraude
0    284315
1       492
Name: count, dtype: int64

Rodando t-SNE 3D...

--------------------------------------------------------------------------------
TSNE(early_exaggeration=12, early_exaggeration_iter=100, exaggeration=1,
     n_components=3, n_iter=1500, n_jobs=7, negative_gradient_method='bh',
     random_state=42, verbose=True)
--------------------------------------------------------------------------------
===> Finding 90 nearest neighbors using Annoy approximate search using euclidean distance...
   --> Time elapsed: 84.68 seconds
===> Calculating affinity matrix...
   --> Time elapsed: 22.13 seconds
===> Calculating PCA-based initialization...
   --> Time elapsed: 0.04 seconds
===> Running optimization with exaggeration=12.00, lr=23733.92 for 100 iterations...
Iteration   5

### ORIGINAL 

In [14]:
resultado = gerar_relatorio_3d(
    RANK=5,
    usar_tsne=False,
    TARGET_COL= TARGET_COL_D,
    THRESHOLD= THRESHOLD_D,
    estado_randomico= estado_randomico_D,

    numero_de_componentes= numero_de_componentes_D,
    inicializacoes_gausianas= inicializacoes_gausianas_D,
    tipo_matriz_covariancia= tipo_matriz_covariancia_D,
    erro_numerico= erro_numerico_D,

    TSNE_PERPLEXITY= TSNE_PERPLEXITY_D,
    TSNE_N_ITER= TSNE_N_ITER_D,
    TSNE_EARLY_EXAGGERATION_ITER= TSNE_EARLY_EXAGGERATION_ITER_D,
    TSNE_EARLY_EXAGGERATION= TSNE_EARLY_EXAGGERATION_D,
    TSNE_EXAGGERATION= TSNE_EXAGGERATION_D,
    TSNE_LEARNING_RATE=TSNE_LEARNING_RATE_D,
    TSNE_METRIC= TSNE_METRIC_D,
    TSNE_INITIALIZATION=TSNE_INITIALIZATION_D,
    TSNE_NEGATIVE_GRADIENT_METHOD=TSNE_NEGATIVE_GRADIENT_METHOD_D,
    TSNE_N_JOBS=TSNE_N_JOBS_D,
    TSNE_VERBOSE=TSNE_VERBOSE_D
)

print(resultado)

Diretório: c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2

Rank Selecionado: 5
Features: V14 vs V17 vs V26

Quantidade usada:
status_fraude
0    284315
1       492
Name: count, dtype: int64

Tabela Cluster x Classe Real:
status_fraude       0    1
row_0                     
0              274307   52
1               10008  440

Cluster identificado como fraude: 1

HTML GERADO COM SUCESSO:
c:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\3d_rank_5_orig.html
{'HTML_PATH': 'c:\\Users\\Vitor Craveiro\\Desktop\\UNESP\\MATÉRIAS FACULDADES\\MATÉRIAS 12 SEMESTRES\\TRABALHO DE CONCLUSAO DE CURSO 2\\3d_rank_5_orig.html', 'Rank': 5, 'Tipo': 'orig', 'Features': ['V14', 'V17', 'V26'], 'AUC_PR': 0.6412307603909041, 'MCC': 0.18978168186861422, 'KS': np.float64(0.8622380077976606), 'Log_Loss': 0.2636993143679451, 'Score_Final': 0.722422}
